# Model Card Generation & Compliance Scorecard

Interactive walkthrough of the governance and compliance pillar.

**What we cover:**
1. Run the full RAI evaluation pipeline
2. Generate an auto-populated model card
3. Build a compliance scorecard with RAG status
4. Feature importance and counterfactual explanations
5. Data minimization analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

from src.pipeline import RAIPipeline

## 1. Run Full RAI Evaluation

In [ ]:
pipeline = RAIPipeline(data_dir='../data/')
result = pipeline.evaluate_hiring_model()

## 2. Compliance Scorecard

In [ ]:
scorecard = result.compliance_scorecard
display(scorecard)
print(f'\nOverall Status: {result.overall_status}')

In [ ]:
# RAG status visualization
rag_colors = {'GREEN': '#2ecc71', 'AMBER': '#f39c12', 'RED': '#e74c3c'}

fig, ax = plt.subplots(figsize=(10, len(scorecard) * 0.5 + 1))
y_pos = range(len(scorecard))
colors = [rag_colors[s] for s in scorecard['status']]
ax.barh(y_pos, [1] * len(scorecard), color=colors, height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row['pillar']} | {row['check']}" for _, row in scorecard.iterrows()], fontsize=8)
ax.set_xlim(0, 1.5)
ax.set_xticks([])
for i, (_, row) in enumerate(scorecard.iterrows()):
    ax.text(1.05, i, row['status'], va='center', fontsize=9, fontweight='bold',
           color=rag_colors[row['status']])
ax.set_title(f'Responsible AI Compliance Scorecard (Overall: {result.overall_status})')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Model Card

In [ ]:
# Display the auto-generated model card
from IPython.display import Markdown
card_md = result.governance_results['model_card_markdown']
display(Markdown(card_md))

## 4. Feature Importance

In [ ]:
global_imp = result.explainability_results['global_importance']
display(global_imp)

fig, ax = plt.subplots(figsize=(8, 4))
imp_sorted = global_imp.sort_values('importance_mean')
ax.barh(imp_sorted['feature'], imp_sorted['importance_mean'],
        xerr=imp_sorted['importance_std'], color='steelblue', capsize=3)
ax.set_xlabel('Permutation Importance (accuracy decrease)')
ax.set_title('Global Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Combined importance (coefficient + correlation)
combined = result.explainability_results['combined_importance']
display(combined)

## 5. Counterfactual Explanations

For rejected candidates: what minimum changes would flip the prediction?

In [ ]:
from src.explainability.counterfactual import CounterfactualGenerator

cfs = result.explainability_results['counterfactuals']
print(f'Counterfactuals generated: {len(cfs)}')

cf_gen = CounterfactualGenerator()
cf_table = cf_gen.summary_table(cfs)
display(cf_table)

In [ ]:
# How many features need to change to flip a rejection?
if len(cf_table) > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    cf_table['n_changes'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_xlabel('Number of Features Changed')
    ax.set_ylabel('Count')
    ax.set_title('Features Changed to Flip Rejection to Hire')
    plt.tight_layout()
    plt.show()

## 6. Data Minimization

In [ ]:
necessity = result.privacy_results['necessity']
display(necessity)

privacy_report = result.privacy_results['privacy_report']
print(f'\nNecessary features: {privacy_report["necessary_features"]}/{privacy_report["total_features"]}')
print(f'Removable features: {privacy_report["removable_features"]}')
if privacy_report['removable_features'] > 0:
    print(f'Candidates for removal: {privacy_report["removable_list"]}')

## 7. Audit Trail

In [ ]:
trail = result.governance_results['audit_trail']
display(trail.to_dataframe())
print(f'\nAudit summary: {trail.summary()}')

## Key Takeaways

- **RED overall status** correctly blocks deployment until ethnicity bias is remediated
- **Model card** auto-populates with metrics, fairness results, and ethical considerations
- **Counterfactuals** provide actionable explanations for rejected candidates
- **Audit trail** creates an immutable record of every evaluation for compliance
- **Production path:** integrate with Azure AI Foundry RAI Dashboard for continuous monitoring